In [1]:
from pathlib import Path

import pandas as pd

In [2]:
DATA_ROOT = Path("../data/raw")

PROJECTS = [
    "eclipse",
    "equinox",
    "lucene",
    "mylyn",
    "pde",
]

In [3]:
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.rstrip(":")
        .str.strip()
    )
    return df

In [4]:
def load_project(project_name):
    data_dir = DATA_ROOT / project_name

    ck = pd.read_csv(
        data_dir / "single-version-ck-oo.csv",
        sep=";",
    )

    change = pd.read_csv(
        data_dir / "change-metrics.csv",
        sep=";",
    )

    ck = clean_columns(ck)
    change = clean_columns(change)

    # Each class should occur exactly once.
    assert not ck["classname"].duplicated().any()
    assert not change["classname"].duplicated().any()

    # CK and change datasets should describe the same classes.
    assert set(ck["classname"]) == set(change["classname"])

    # Verify that the bug labels agree between the two datasets.
    label_check = ck[["classname", "bugs"]].merge(
        change[["classname", "bugs"]],
        on="classname",
        suffixes=("_ck", "_change"),
        validate="one_to_one",
    )

    assert (
        label_check["bugs_ck"] ==
        label_check["bugs_change"]
    ).all()

    # Keep the bug labels from CK and remove duplicate labels
    # from the change-metrics dataset.
    change_features = change.drop(
        columns=[
            "bugs",
            "nonTrivialBugs",
            "majorBugs",
            "criticalBugs",
            "highPriorityBugs",
        ]
    )

    merged = ck.merge(
        change_features,
        on="classname",
        how="inner",
        validate="one_to_one",
    )

    # Needed for leave-one-project-out evaluation later.
    merged["project"] = project_name

    # Binary prediction target.
    merged["defective"] = (
        merged["bugs"] > 0
    ).astype(int)

    return merged

In [5]:
project_dfs = [
    load_project(project)
    for project in PROJECTS
]

all_df = pd.concat(
    project_dfs,
    ignore_index=True,
)

In [6]:
all_df.groupby("project")["defective"].agg(
    observations="size",
    defective="sum",
    defect_rate="mean",
)

,observations,defective,defect_rate
project,,,
eclipse,997,206,0.206620
equinox,324,129,0.398148
lucene,691,64,0.092619
mylyn,1862,245,0.131579
pde,1497,209,0.139613


In [7]:
CK_FEATURES = [
    "wmc",
    "cbo",
    "dit",
    "lcom",
    "rfc",
    "noc",
    "numberOfLinesOfCode",
    "numberOfMethods",
]

CHANGE_FEATURES = [
    "numberOfVersionsUntil",
    "numberOfAuthorsUntil",

    "linesAddedUntil",
    "maxLinesAddedUntil",
    "avgLinesAddedUntil",

    "linesRemovedUntil",
    "maxLinesRemovedUntil",
    "avgLinesRemovedUntil",

    "codeChurnUntil",
    "maxCodeChurnUntil",
    "avgCodeChurnUntil",

    "ageWithRespectTo",
    "weightedAgeWithRespectTo",
]

FEATURE_SETS = {
    "CK": CK_FEATURES,
    "Change": CHANGE_FEATURES,
    "CK + Change": CK_FEATURES + CHANGE_FEATURES,
}

In [8]:
for name, features in FEATURE_SETS.items():
    missing = set(features) - set(all_df.columns)
    print(f"{name}: {len(features)} features, missing = {missing}")

CK: 8 features, missing = set()
Change: 13 features, missing = set()
CK + Change: 21 features, missing = set()


In [9]:
all_features = CK_FEATURES + CHANGE_FEATURES

all_df[all_features].isna().sum().sort_values(ascending=False)

wmc                         0
cbo                         0
dit                         0
lcom                        0
rfc                         0
noc                         0
numberOfLinesOfCode         0
numberOfMethods             0
numberOfVersionsUntil       0
numberOfAuthorsUntil        0
linesAddedUntil             0
maxLinesAddedUntil          0
avgLinesAddedUntil          0
linesRemovedUntil           0
maxLinesRemovedUntil        0
avgLinesRemovedUntil        0
codeChurnUntil              0
maxCodeChurnUntil           0
avgCodeChurnUntil           0
ageWithRespectTo            0
weightedAgeWithRespectTo    0
dtype: int64

In [10]:
X = all_df[CK_FEATURES]
y = all_df["defective"]

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head())

X shape: (5371, 8)
y shape: (5371,)


,wmc,cbo,dit,lcom,rfc,noc,numberOfLinesOfCode,numberOfMethods
0,20.0,9,2,15,34.0,0,122.0,6.0
1,1.0,1,1,0,1.0,0,4.0,1.0
2,176.0,114,1,190,156.0,6,484.0,20.0
3,12.0,5,6,10,18.0,0,33.0,5.0
4,115.0,23,2,820,174.0,0,673.0,41.0


0    0
1    0
2    1
3    0
4    0
Name: defective, dtype: int64

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [12]:
def make_logistic_model():
    return Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ])

In [13]:
def evaluate_lopo(df, features, model_factory):
    results = []

    projects = sorted(df["project"].unique())

    for test_project in projects:
        train = df[df["project"] != test_project]
        test = df[df["project"] == test_project]

        X_train = train[features]
        y_train = train["defective"]

        X_test = test[features]
        y_test = test["defective"]

        model = model_factory()

        model.fit(
            X_train,
            y_train,
        )

        probabilities = model.predict_proba(X_test)[:, 1]

        results.append({
            "test_project": test_project,
            "n_train": len(train),
            "n_test": len(test),
            "defect_rate": y_test.mean(),

            "roc_auc": roc_auc_score(
                y_test,
                probabilities,
            ),

            "pr_auc": average_precision_score(
                y_test,
                probabilities,
            ),

            "brier": brier_score_loss(
                y_test,
                probabilities,
            ),
        })

    return pd.DataFrame(results)

In [14]:
ck_results = evaluate_lopo(
    all_df,
    CK_FEATURES,
    make_logistic_model,
)

ck_results

,test_project,n_train,n_test,defect_rate,roc_auc,pr_auc,brier
0,eclipse,4374,997,0.206620,0.687725,0.491457,0.196647
1,equinox,5047,324,0.398148,0.733612,0.668306,0.210100
2,lucene,4680,691,0.092619,0.612864,0.191487,0.205355
3,mylyn,3509,1862,0.131579,0.697674,0.309570,0.191643
4,pde,3874,1497,0.139613,0.700641,0.309397,0.219604


In [15]:
ck_results[
    ["roc_auc", "pr_auc", "brier"]
].mean()

roc_auc    0.686503
pr_auc     0.394043
brier      0.204670
dtype: float64

In [16]:
change_results = evaluate_lopo(
    all_df,
    CHANGE_FEATURES,
    make_logistic_model,
)

change_results

,test_project,n_train,n_test,defect_rate,roc_auc,pr_auc,brier
0,eclipse,4374,997,0.206620,0.799915,0.607041,0.283170
1,equinox,5047,324,0.398148,0.690757,0.646092,0.212743
2,lucene,4680,691,0.092619,0.765737,0.391950,0.146432
3,mylyn,3509,1862,0.131579,0.689151,0.297826,0.184004
4,pde,3874,1497,0.139613,0.725454,0.327465,0.219767


In [17]:
change_results[
    ["roc_auc", "pr_auc", "brier"]
].mean()

roc_auc    0.734203
pr_auc     0.454075
brier      0.209223
dtype: float64

In [20]:
combined_results = evaluate_lopo(
    all_df,
    CK_FEATURES + CHANGE_FEATURES,
    make_logistic_model,
)

combined_results

,test_project,n_train,n_test,defect_rate,roc_auc,pr_auc,brier
0,eclipse,4374,997,0.206620,0.767033,0.575967,0.243280
1,equinox,5047,324,0.398148,0.658000,0.639498,0.213942
2,lucene,4680,691,0.092619,0.735696,0.320751,0.148611
3,mylyn,3509,1862,0.131579,0.717770,0.325906,0.181204
4,pde,3874,1497,0.139613,0.743989,0.344584,0.224919


In [21]:
change_results[
    ["roc_auc", "pr_auc", "brier"]
].mean()

roc_auc    0.734203
pr_auc     0.454075
brier      0.209223
dtype: float64

In [22]:
comparison = pd.DataFrame({
    "CK only": ck_results[
        ["roc_auc", "pr_auc", "brier"]
    ].mean(),

    "Change only": change_results[
        ["roc_auc", "pr_auc", "brier"]
    ].mean(),

    "CK + Change": combined_results[
        ["roc_auc", "pr_auc", "brier"]
    ].mean(),
})

comparison

,CK only,Change only,CK + Change
roc_auc,0.686503,0.734203,0.724498
pr_auc,0.394043,0.454075,0.441341
brier,0.204670,0.209223,0.202391


In [23]:
from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier,
)

In [24]:
def make_random_forest():
    return RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )


def make_extra_trees():
    return ExtraTreesClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )

In [25]:
MODELS = {
    "Logistic Regression": make_logistic_model,
    "Random Forest": make_random_forest,
    "Extra Trees": make_extra_trees,
}

In [30]:
experiment_results = []

for model_name, model_factory in MODELS.items():
    for feature_name, features in FEATURE_SETS.items():

        print(f"Running: {model_name} + {feature_name}")

        result = evaluate_lopo(
            all_df,
            features,
            model_factory,
        )

        experiment_results.append({
            "model": model_name,
            "features": feature_name,
            "roc_auc": result["roc_auc"].mean(),
            "pr_auc": result["pr_auc"].mean(),
            "brier": result["brier"].mean(),
        })

        print("  done")

experiment_results = pd.DataFrame(experiment_results)
experiment_results

Running: Logistic Regression + CK
  done
Running: Logistic Regression + Change
  done
Running: Logistic Regression + CK + Change
  done
Running: Random Forest + CK
  done
Running: Random Forest + Change
  done
Running: Random Forest + CK + Change
  done
Running: Extra Trees + CK
  done
Running: Extra Trees + Change
  done
Running: Extra Trees + CK + Change
  done


,model,features,roc_auc,pr_auc,brier
0,Logistic Regression,CK,0.686503,0.394043,0.204670
1,Logistic Regression,Change,0.734203,0.454075,0.209223
2,Logistic Regression,CK + Change,0.724498,0.441341,0.202391
3,Random Forest,CK,0.656872,0.356510,0.165111
4,Random Forest,Change,0.698563,0.357572,0.175236
5,Random Forest,CK + Change,0.722437,0.421754,0.157174
6,Extra Trees,CK,0.660269,0.333412,0.155726
7,Extra Trees,Change,0.700469,0.364757,0.153530
8,Extra Trees,CK + Change,0.724118,0.416315,0.141205


In [31]:
experiment_results.sort_values(
    "roc_auc",
    ascending=False,
)

,model,features,roc_auc,pr_auc,brier
1,Logistic Regression,Change,0.734203,0.454075,0.209223
2,Logistic Regression,CK + Change,0.724498,0.441341,0.202391
8,Extra Trees,CK + Change,0.724118,0.416315,0.141205
5,Random Forest,CK + Change,0.722437,0.421754,0.157174
7,Extra Trees,Change,0.700469,0.364757,0.153530
4,Random Forest,Change,0.698563,0.357572,0.175236
0,Logistic Regression,CK,0.686503,0.394043,0.204670
6,Extra Trees,CK,0.660269,0.333412,0.155726
3,Random Forest,CK,0.656872,0.356510,0.165111


In [32]:
experiment_results.sort_values(
    "pr_auc",
    ascending=False,
)

,model,features,roc_auc,pr_auc,brier
1,Logistic Regression,Change,0.734203,0.454075,0.209223
2,Logistic Regression,CK + Change,0.724498,0.441341,0.202391
5,Random Forest,CK + Change,0.722437,0.421754,0.157174
8,Extra Trees,CK + Change,0.724118,0.416315,0.141205
0,Logistic Regression,CK,0.686503,0.394043,0.204670
7,Extra Trees,Change,0.700469,0.364757,0.153530
4,Random Forest,Change,0.698563,0.357572,0.175236
3,Random Forest,CK,0.656872,0.356510,0.165111
6,Extra Trees,CK,0.660269,0.333412,0.155726


In [34]:
def evaluate_lopo(df, features, model_factory, verbose=False):
    results = []

    projects = sorted(df["project"].unique())

    for i, test_project in enumerate(projects, start=1):
        if verbose:
            print(
                f"[{i}/{len(projects)}] Testing on {test_project}...",
                flush=True,
            )

        train = df[df["project"] != test_project]
        test = df[df["project"] == test_project]

        X_train = train[features]
        y_train = train["defective"]

        X_test = test[features]
        y_test = test["defective"]

        model = model_factory()

        model.fit(X_train, y_train)

        probabilities = model.predict_proba(X_test)[:, 1]

        results.append({
            "test_project": test_project,
            "n_train": len(train),
            "n_test": len(test),
            "defect_rate": y_test.mean(),
            "roc_auc": roc_auc_score(y_test, probabilities),
            "pr_auc": average_precision_score(y_test, probabilities),
            "brier": brier_score_loss(y_test, probabilities),
        })

        if verbose:
            print("      ✓ Done", flush=True)

    return pd.DataFrame(results)

In [35]:
print("=== Random Forest ===")

rf_combined = evaluate_lopo(
    all_df,
    CK_FEATURES + CHANGE_FEATURES,
    make_random_forest,
    verbose=True,
)

print("\n=== Extra Trees ===")

et_combined = evaluate_lopo(
    all_df,
    CK_FEATURES + CHANGE_FEATURES,
    make_extra_trees,
    verbose=True,
)

print("\n=== Results ===")

display(rf_combined)
display(et_combined)

=== Random Forest ===
[1/5] Testing on eclipse...
      ✓ Done
[2/5] Testing on equinox...
      ✓ Done
[3/5] Testing on lucene...
      ✓ Done
[4/5] Testing on mylyn...
      ✓ Done
[5/5] Testing on pde...
      ✓ Done

=== Extra Trees ===
[1/5] Testing on eclipse...
      ✓ Done
[2/5] Testing on equinox...
      ✓ Done
[3/5] Testing on lucene...
      ✓ Done
[4/5] Testing on mylyn...
      ✓ Done
[5/5] Testing on pde...
      ✓ Done

=== Results ===


,test_project,n_train,n_test,defect_rate,roc_auc,pr_auc,brier
0,eclipse,4374,997,0.206620,0.735946,0.512019,0.198067
1,equinox,5047,324,0.398148,0.744941,0.662295,0.220672
2,lucene,4680,691,0.092619,0.730724,0.298857,0.101749
3,mylyn,3509,1862,0.131579,0.686851,0.334306,0.116840
4,pde,3874,1497,0.139613,0.713723,0.301293,0.148540


,test_project,n_train,n_test,defect_rate,roc_auc,pr_auc,brier
0,eclipse,4374,997,0.206620,0.771780,0.550877,0.142526
1,equinox,5047,324,0.398148,0.747088,0.661649,0.259400
2,lucene,4680,691,0.092619,0.709567,0.243387,0.084011
3,mylyn,3509,1862,0.131579,0.672057,0.319600,0.105739
4,pde,3874,1497,0.139613,0.720099,0.306061,0.114350


In [36]:
from sklearn.inspection import permutation_importance

In [37]:
def lopo_permutation_importance(
    df,
    features,
    model_factory,
):
    results = []

    projects = sorted(df["project"].unique())

    for i, test_project in enumerate(projects, start=1):
        print(
            f"[{i}/{len(projects)}] {test_project}...",
            flush=True,
        )

        train = df[df["project"] != test_project]
        test = df[df["project"] == test_project]

        X_train = train[features]
        y_train = train["defective"]

        X_test = test[features]
        y_test = test["defective"]

        model = model_factory()
        model.fit(X_train, y_train)

        importance = permutation_importance(
            model,
            X_test,
            y_test,
            scoring="roc_auc",
            n_repeats=10,
            random_state=42,
            n_jobs=-1,
        )

        for feature, mean, std in zip(
            features,
            importance.importances_mean,
            importance.importances_std,
        ):
            results.append({
                "project": test_project,
                "feature": feature,
                "importance": mean,
                "std": std,
            })

        print("    ✓ Done")

    return pd.DataFrame(results)

In [38]:
rf_importance = lopo_permutation_importance(
    all_df,
    CK_FEATURES + CHANGE_FEATURES,
    make_random_forest,
)

[1/5] eclipse...
    ✓ Done
[2/5] equinox...
    ✓ Done
[3/5] lucene...
    ✓ Done
[4/5] mylyn...
    ✓ Done
[5/5] pde...
    ✓ Done


In [39]:
rf_importance_summary = (
    rf_importance
    .groupby("feature")
    .agg(
        mean_importance=("importance", "mean"),
        std_across_projects=("importance", "std"),
    )
    .sort_values(
        "mean_importance",
        ascending=False,
    )
)

rf_importance_summary

,mean_importance,std_across_projects
feature,,
linesAddedUntil,0.022624,0.018572
cbo,0.013649,0.007581
numberOfLinesOfCode,0.009979,0.005694
maxCodeChurnUntil,0.009520,0.009885
rfc,0.009203,0.014989
numberOfVersionsUntil,0.007715,0.009990
wmc,0.006975,0.004486
maxLinesRemovedUntil,0.005994,0.004131
codeChurnUntil,0.005088,0.003133


In [40]:
REDUCED_FEATURES = [
    # Static
    "cbo",
    "numberOfLinesOfCode",
    "rfc",
    "wmc",

    # Process
    "linesAddedUntil",
    "maxCodeChurnUntil",
    "numberOfVersionsUntil",
    "maxLinesRemovedUntil",
    "codeChurnUntil",
]

In [41]:
rf_reduced = evaluate_lopo(
    all_df,
    REDUCED_FEATURES,
    make_random_forest,
    verbose=True,
)

rf_reduced

[1/5] Testing on eclipse...
      ✓ Done
[2/5] Testing on equinox...
      ✓ Done
[3/5] Testing on lucene...
      ✓ Done
[4/5] Testing on mylyn...
      ✓ Done
[5/5] Testing on pde...
      ✓ Done


,test_project,n_train,n_test,defect_rate,roc_auc,pr_auc,brier
0,eclipse,4374,997,0.206620,0.767935,0.565054,0.200850
1,equinox,5047,324,0.398148,0.756788,0.666961,0.216963
2,lucene,4680,691,0.092619,0.684198,0.268514,0.112054
3,mylyn,3509,1862,0.131579,0.656240,0.303883,0.120115
4,pde,3874,1497,0.139613,0.693405,0.267704,0.163725


In [42]:
reduced_comparison = pd.DataFrame({
    "RF Full (21)": rf_combined[
        ["roc_auc", "pr_auc", "brier"]
    ].mean(),

    "RF Reduced (9)": rf_reduced[
        ["roc_auc", "pr_auc", "brier"]
    ].mean(),
})

reduced_comparison

,RF Full (21),RF Reduced (9)
roc_auc,0.722437,0.711713
pr_auc,0.421754,0.414423
brier,0.157174,0.162741


In [43]:
def feature_ablation(
    df,
    features,
    model_factory,
):
    results = []

    print("Evaluating full feature set...")
    full = evaluate_lopo(
        df,
        features,
        model_factory,
        verbose=False,
    )

    baseline_roc = full["roc_auc"].mean()
    baseline_pr = full["pr_auc"].mean()
    baseline_brier = full["brier"].mean()

    print(
        f"Baseline: ROC={baseline_roc:.4f}, "
        f"PR={baseline_pr:.4f}, "
        f"Brier={baseline_brier:.4f}\n"
    )

    for i, removed_feature in enumerate(features, start=1):
        print(
            f"[{i}/{len(features)}] Removing {removed_feature}...",
            flush=True,
        )

        reduced_features = [
            feature
            for feature in features
            if feature != removed_feature
        ]

        result = evaluate_lopo(
            df,
            reduced_features,
            model_factory,
            verbose=False,
        )

        roc = result["roc_auc"].mean()
        pr = result["pr_auc"].mean()
        brier = result["brier"].mean()

        results.append({
            "removed_feature": removed_feature,

            "roc_auc": roc,
            "delta_roc": roc - baseline_roc,

            "pr_auc": pr,
            "delta_pr": pr - baseline_pr,

            "brier": brier,
            "delta_brier": brier - baseline_brier,
        })

        print(
            f"    ROC Δ={roc - baseline_roc:+.4f}, "
            f"PR Δ={pr - baseline_pr:+.4f}, "
            f"Brier Δ={brier - baseline_brier:+.4f}"
        )

    return pd.DataFrame(results)

In [44]:
ALL_FEATURES = CK_FEATURES + CHANGE_FEATURES

rf_ablation = feature_ablation(
    all_df,
    ALL_FEATURES,
    make_random_forest,
)

Evaluating full feature set...
Baseline: ROC=0.7224, PR=0.4218, Brier=0.1572

[1/21] Removing wmc...
    ROC Δ=+0.0002, PR Δ=+0.0044, Brier Δ=+0.0005
[2/21] Removing cbo...
    ROC Δ=-0.0086, PR Δ=-0.0275, Brier Δ=+0.0033
[3/21] Removing dit...
    ROC Δ=+0.0062, PR Δ=+0.0026, Brier Δ=+0.0019
[4/21] Removing lcom...
    ROC Δ=-0.0049, PR Δ=-0.0047, Brier Δ=+0.0010
[5/21] Removing rfc...
    ROC Δ=+0.0011, PR Δ=-0.0101, Brier Δ=+0.0011
[6/21] Removing noc...
    ROC Δ=+0.0040, PR Δ=+0.0002, Brier Δ=-0.0002
[7/21] Removing numberOfLinesOfCode...
    ROC Δ=+0.0002, PR Δ=-0.0047, Brier Δ=+0.0009
[8/21] Removing numberOfMethods...
    ROC Δ=-0.0037, PR Δ=-0.0005, Brier Δ=+0.0004
[9/21] Removing numberOfVersionsUntil...
    ROC Δ=+0.0076, PR Δ=+0.0036, Brier Δ=-0.0023
[10/21] Removing numberOfAuthorsUntil...
    ROC Δ=+0.0006, PR Δ=+0.0011, Brier Δ=-0.0007
[11/21] Removing linesAddedUntil...
    ROC Δ=-0.0017, PR Δ=-0.0000, Brier Δ=-0.0004
[12/21] Removing maxLinesAddedUntil...
    ROC Δ=+0.

In [45]:
REMOVE_CANDIDATES = [
    "numberOfVersionsUntil",
    "numberOfAuthorsUntil",
    "noc",
    "maxCodeChurnUntil",
]

In [46]:
ALL_FEATURES = CK_FEATURES + CHANGE_FEATURES

ABLATION_REDUCED_FEATURES = [
    feature
    for feature in ALL_FEATURES
    if feature not in REMOVE_CANDIDATES
]

print("Full:", len(ALL_FEATURES))
print("Reduced:", len(ABLATION_REDUCED_FEATURES))
print(ABLATION_REDUCED_FEATURES)

Full: 21
Reduced: 17
['wmc', 'cbo', 'dit', 'lcom', 'rfc', 'numberOfLinesOfCode', 'numberOfMethods', 'linesAddedUntil', 'maxLinesAddedUntil', 'avgLinesAddedUntil', 'linesRemovedUntil', 'maxLinesRemovedUntil', 'avgLinesRemovedUntil', 'codeChurnUntil', 'avgCodeChurnUntil', 'ageWithRespectTo', 'weightedAgeWithRespectTo']


In [47]:
rf_ablation_reduced = evaluate_lopo(
    all_df,
    ABLATION_REDUCED_FEATURES,
    make_random_forest,
    verbose=True,
)

rf_ablation_reduced

[1/5] Testing on eclipse...
      ✓ Done
[2/5] Testing on equinox...
      ✓ Done
[3/5] Testing on lucene...
      ✓ Done
[4/5] Testing on mylyn...
      ✓ Done
[5/5] Testing on pde...
      ✓ Done


,test_project,n_train,n_test,defect_rate,roc_auc,pr_auc,brier
0,eclipse,4374,997,0.206620,0.718124,0.483507,0.199840
1,equinox,5047,324,0.398148,0.750467,0.676980,0.215876
2,lucene,4680,691,0.092619,0.737889,0.306741,0.100059
3,mylyn,3509,1862,0.131579,0.684742,0.325480,0.117867
4,pde,3874,1497,0.139613,0.706795,0.300273,0.149942


In [48]:
pd.DataFrame({
    "RF Full (21)": rf_combined[
        ["roc_auc", "pr_auc", "brier"]
    ].mean(),

    "RF Reduced (9)": rf_reduced[
        ["roc_auc", "pr_auc", "brier"]
    ].mean(),

    "RF Ablation Reduced (17)": rf_ablation_reduced[
        ["roc_auc", "pr_auc", "brier"]
    ].mean(),
})

,RF Full (21),RF Reduced (9),RF Ablation Reduced (17)
roc_auc,0.722437,0.711713,0.719604
pr_auc,0.421754,0.414423,0.418596
brier,0.157174,0.162741,0.156717
